# DistilBERT News Classification with Hugging Face

This notebook demonstrates an end-to-end NLP workflow for multi-class news classification using the AG News dataset and a fine-tuned DistilBERT transformer model.

## Project Summary

In this notebook, I:

- Load and inspect the AG News dataset
- Prepare label mappings for a four-class classification problem
- Tokenize article text with a Hugging Face tokenizer
- Fine-tune a DistilBERT sequence classification model
- Evaluate model performance
- Run inference on sample headlines
- Discuss business relevance and practical next steps

## Why This Project Matters

Text classification is a foundational NLP capability with direct business value. A workflow like this can support content tagging, document routing, customer-ticket triage, media monitoring, and knowledge management.

**Originally developed as graduate coursework and refined for portfolio presentation.**

In [ ]:
# If running in Google Colab, install the required libraries (run once).
# This cell may take a few minutes the first time.
# You can comment it out if the libraries are already installed.

!pip install -q transformers datasets accelerate evaluate

In [ ]:
# Import standard libraries and Hugging Face tools
import numpy as np
import pandas as pd

from datasets import load_dataset
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline,
)

In [ ]:
# Load the AG News dataset from the Hugging Face hub
# The dataset has 'train' and 'test' splits with 'text' and 'label' fields.

dataset = load_dataset("ag_news")

# Peek at the first two training examples to understand structure
dataset['train'][0], dataset['train'][1]

In [ ]:
# Extract label names and build id2label / label2id mappings
labels = dataset['train'].features['label'].names
id2label = {i: label for i, label in enumerate(labels)}
label2id = {label: i for i, label in enumerate(labels)}

print("Labels:", labels)
print("id2label:", id2label)
print("label2id:", label2id)

In [ ]:
# Choose a compact pre-trained model checkpoint
MODEL_CHECKPOINT = "distilbert-base-uncased"

# Load the matching tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_function(examples):
    """Tokenize raw text into input_ids and attention_mask.

    We use padding='max_length' for simplicity in this assignment and truncation=True
    to avoid exceedingly long sequences. The `max_length` is a tunable parameter that
    trades off speed vs. ability to handle long articles.
    """
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128,  # Tunable: sequence length
    )

# Apply tokenization in batched mode for speed
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Prepare dataset for PyTorch models by removing raw text and renaming label
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

tokenized_datasets

In [ ]:
# Load a sequence classification model with the correct number of labels
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

In [ ]:
# Use accuracy as the primary evaluation metric for this assignment
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    """Compute accuracy from model logits and true labels.

    Hugging Face's Trainer passes (logits, labels), so we take argmax over
    the last dimension to get predicted class IDs, then compute accuracy.
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [ ]:
# Training hyperparameters (all tunable)
NUM_EPOCHS = 2          # Small for demo; increasing can improve performance
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01

training_args = TrainingArguments(
    output_dir="seas6505_hw4_1_agnews_model",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    logging_dir="./logs_hw4_1",
    logging_steps=50,
    load_best_model_at_end=True,
)

In [ ]:
# Create a Trainer object that wraps the training loop
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

# Fine-tune the model
train_result = trainer.train()

# Evaluate on the held-out test set
eval_results = trainer.evaluate()
eval_results

In [ ]:
# Build a text-classification pipeline for easy inference
clf_pipeline = pipeline(
    "text-classification",
    model=trainer.model,
    tokenizer=tokenizer,
)

# Try the pipeline on a few sample headlines
sample_texts = [
    "Federal Reserve announces interest rate hike amid inflation concerns.",
    "Local team wins championship in stunning overtime victory.",
    "Tech company releases new smartphone with AI-powered camera.",
]

for text in sample_texts:
    print(text)
    print(clf_pipeline(text), "\n")

## Technical Analysis

This project applies a modern transformer-based NLP workflow to the AG News classification task using the Hugging Face ecosystem. I loaded the dataset with `datasets.load_dataset`, reviewed its structure, and created `id2label` and `label2id` mappings so model outputs remain interpretable.

For preprocessing, I used the `distilbert-base-uncased` tokenizer and defined a tokenization function that pads and truncates each example to a fixed `max_length=128`. This keeps the workflow efficient while preserving enough context for short news articles. After tokenization, I removed the raw text field, renamed the target column to `labels`, and formatted the dataset for PyTorch.

I then fine-tuned `AutoModelForSequenceClassification` with the correct number of output classes and used Hugging Face's `Trainer` API to manage training and evaluation. The notebook exposes key hyperparameters such as number of epochs, batch size, learning rate, warmup ratio, and weight decay so the workflow can be tuned further. Accuracy is used as the primary evaluation metric, and the final model is wrapped in a text-classification pipeline for simple inference on new headlines.

## Portfolio Framing

This notebook demonstrates practical experience with:

- Transformer-based NLP
- Hugging Face tokenization and fine-tuning workflows
- PyTorch-ready dataset preparation
- Model evaluation and inference
- Translating technical output into business use cases

## Business Relevance and Recommendations

Although the AG News dataset is a benchmark dataset, the workflow is directly transferable to real business problems involving unstructured text. A similar pipeline could be used to classify support tickets, categorize inbound emails, route claims or cases, tag documents, or monitor external content streams.

A strong next step would be to replace AG News with a domain-specific labeled dataset and compare model performance against simpler baselines. I would also extend evaluation beyond accuracy to include precision, recall, and F1 score, especially in settings with class imbalance. Once validated, the model could be deployed behind an API or internal tool to support faster routing, better analytics, and more consistent decision-making.
